# DeepSeek Coder 6.7B (Q4) trên Colab T4 bằng Vulkan

Chọn **Runtime → Change runtime type → T4 GPU**, rồi chạy lần lượt từ trên xuống. Notebook cài Nim 2.2.10 và Vulkan, clone repo kèm submodule, tải `model.nimq` cùng tokenizer từ [bobbydeveloper2014/deepseek-coder-6.7b-nimq](https://huggingface.co/bobbydeveloper2014/deepseek-coder-6.7b-nimq), build `nim_inference`, rồi chạy inference trên Vulkan. Tokenizer và chat template chạy thuần Nim; không cần `transformers`.

Tensor Core của T4 được dùng giống Codestral: các GEMM prefill (Q/K/V/O, gate/up/down của cả 32 layer) chạy kernel `KHR_cooperative_matrix`. Kernel này giải nén Q4 thành FP16 theo từng tile và cộng dồn FP32. Bộ autotune kiểm tra parity với kernel tham chiếu ngay trên GPU, đo ở M=64, rồi mới chọn. Decode (1 token/bước) là GEMV bị giới hạn bởi băng thông, nên dùng kernel scalar như Codestral.

Sửa `PROMPT`, `MAX_NEW_TOKENS`, `TEMPERATURE` ở cell inference.

In [1]:
# 1) Kiểm tra runtime GPU và cài các gói hệ thống, gồm Vulkan runtime và shader compiler (glslc).
import os, subprocess, sys
from pathlib import Path

assert Path('/content').exists(), 'Notebook này cần chạy trên Google Colab.'
try:
    print(subprocess.check_output(['nvidia-smi', '--query-gpu=name,driver_version,memory.total', '--format=csv,noheader'], text=True))
except Exception as e:
    raise RuntimeError('Chưa có GPU NVIDIA. Chọn Runtime → Change runtime type → T4 GPU rồi chạy lại.') from e

subprocess.run(['apt-get','update','-qq'], check=True)
subprocess.run(['apt-get','install','-y','-qq','build-essential','git','curl','ca-certificates','xz-utils','pkg-config','time','libvulkan1','libvulkan-dev','vulkan-tools','glslc','libegl1','libglx0','libx11-6','libxext6','libx11-xcb1','libxcb1','libxcb-dri3-0','libxcb-present0','libxfixes3','libxrandr2','libdrm2','libexpat1','libwayland-client0'], check=True)

Tesla T4, 580.82.07, 15360 MiB



CompletedProcess(args=['apt-get', 'install', '-y', '-qq', 'build-essential', 'git', 'curl', 'ca-certificates', 'xz-utils', 'pkg-config', 'time', 'libvulkan1', 'libvulkan-dev', 'vulkan-tools', 'glslc', 'libegl1', 'libglx0', 'libx11-6', 'libxext6', 'libx11-xcb1', 'libxcb1', 'libxcb-dri3-0', 'libxcb-present0', 'libxfixes3', 'libxrandr2', 'libdrm2', 'libexpat1', 'libwayland-client0'], returncode=0)

In [2]:
# 2) Cài Nim 2.2.10 chính chủ và xác minh checksum (tokenizer thuần Nim).
import hashlib, urllib.request, tarfile, os, subprocess
from pathlib import Path

NIM_VERSION = '2.2.10'
TOOLS = Path('/content/tools'); TOOLS.mkdir(exist_ok=True)
archive = TOOLS / f'nim-{NIM_VERSION}-linux_x64.tar.xz'
url = f'https://nim-lang.org/download/nim-{NIM_VERSION}-linux_x64.tar.xz'

# Tải file với User-Agent giả lập trình duyệt để tránh lỗi 403
if not archive.exists():
    req = urllib.request.Request(
        url,
        headers={'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'}
    )
    with urllib.request.urlopen(req) as response, open(archive, 'wb') as out_file:
        out_file.write(response.read())

# Xác minh checksum
expected = '0a3a38752e97e9d44aa479b3a7b37336dfe0176daf22ee5b5218ad0991ecd211'
actual = hashlib.sha256(archive.read_bytes()).hexdigest()
assert actual == expected, f'Nim checksum mismatch: {actual}'

# Giải nén
with tarfile.open(archive, 'r:xz') as tf:
    tf.extractall(TOOLS, filter='data')

# Cấu hình PATH
NIM_HOME = TOOLS / f'nim-{NIM_VERSION}'
os.environ['PATH'] = f"{NIM_HOME}/bin:" + os.environ['PATH']

# Kiểm tra phiên bản
subprocess.run(['nim', '--version'], check=True)

CompletedProcess(args=['nim', '--version'], returncode=0)

In [3]:
# 3) Clone repository bản main mới nhất và tải submodule đã ghim trong repo.
import os, subprocess
from pathlib import Path

URL = 'https://github.com/bobbyshop-vui/nimformers-framework.git'
BRANCH = 'main'
REPO = Path('/content/nimformers-framework')
if not REPO.exists():
    subprocess.run(['git', 'clone', '--branch', BRANCH, URL, str(REPO)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO), 'fetch', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO), 'checkout', '-B', BRANCH, f'origin/{BRANCH}'], check=True)
subprocess.run(['git', '-C', str(REPO), 'submodule', 'sync', '--recursive'], check=True)
subprocess.run(['git', '-C', str(REPO), 'submodule', 'update', '--init', '--recursive'], check=True)
os.chdir(REPO)
print('Repo:', REPO, '| branch:', BRANCH)
print(subprocess.check_output(['git', 'log', '--oneline', '-1'], text=True))

Repo: /content/nimformers-framework | branch: main
1b0b9a9 Fix lifted RoPE upload classification and positional replay



In [4]:
# 4) Cài Vulkan + kiểm tra ICD NVIDIA và xác nhận dGPU (không chấp nhận lavapipe/CPU)
import subprocess, os, re
from pathlib import Path

# --- Bước 1: Cài vulkan-tools ---
print("[1/4] Cài vulkan-tools...")
subprocess.run(['apt-get', 'update', '-qq'], check=True)
subprocess.run(['apt-get', 'install', '-y', '-qq', 'vulkan-tools'], check=True)

# --- Bước 2: Xác định phiên bản driver NVIDIA và cài libnvidia-gl tương ứng ---
print("[2/4] Phát hiện phiên bản driver NVIDIA...")
smi = subprocess.check_output(['nvidia-smi'], text=True)
m = re.search(r'Driver Version:\s*(\d+)\.', smi)
assert m, "Không đọc được Driver Version từ nvidia-smi"
driver_major = m.group(1)
print(f"    -> Driver major: {driver_major}")

pkg = f'libnvidia-gl-{driver_major}'
print(f"[2/4] Cài {pkg}...")
subprocess.run(['apt-get', 'install', '-y', '-qq', pkg], check=True)

# --- Bước 3: Đảm bảo file ICD NVIDIA có thể được loader tìm thấy ---
print("[3/4] Cấu hình Vulkan ICD...")
Path('/usr/share/vulkan/icd.d').mkdir(parents=True, exist_ok=True)
candidates = [
    '/etc/vulkan/icd.d/nvidia_icd.json',
    '/usr/share/vulkan/icd.d/nvidia_icd.json',
    f'/usr/lib/x86_64-linux-gnu/libnvidia-gl-{driver_major}/nvidia_icd.json',
]
src = next((c for c in candidates if Path(c).exists()), None)
if src and src != '/usr/share/vulkan/icd.d/nvidia_icd.json':
    subprocess.run(['ln', '-sf', src, '/usr/share/vulkan/icd.d/nvidia_icd.json'], check=True)
    print(f"    -> symlink: {src} -> /usr/share/vulkan/icd.d/nvidia_icd.json")
else:
    print(f"    -> ICD có sẵn: {src}")

# --- Bước 4: Chạy kiểm tra ---
print("[4/4] Kiểm tra Vulkan...")
subprocess.run(['vulkaninfo', '--summary'], check=True,
               stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
summary = subprocess.check_output(['vulkaninfo', '--summary'],
                                   text=True, stderr=subprocess.STDOUT)
print(''.join(line for line in summary.splitlines()
              if any(k in line.lower() for k in ('device name', 'device type', 'driver name'))))

assert 'NVIDIA' in summary.upper(), \
    'Không thấy NVIDIA Vulkan ICD. Runtime GPU hiện tại không expose Vulkan graphics.'
assert 'DISCRETE_GPU' in summary.upper(), \
    'Không tìm thấy NVIDIA dGPU qua Vulkan; dừng để tránh fallback CPU/iGPU.'

print("✅ Vulkan OK: NVIDIA dGPU đã được expose.")

[1/4] Cài vulkan-tools...
[2/4] Phát hiện phiên bản driver NVIDIA...
    -> Driver major: 580
[2/4] Cài libnvidia-gl-580...
[3/4] Cấu hình Vulkan ICD...
    -> ICD có sẵn: /usr/share/vulkan/icd.d/nvidia_icd.json
[4/4] Kiểm tra Vulkan...

✅ Vulkan OK: NVIDIA dGPU đã được expose.


In [5]:
# 5) Tải model Q4 (~4.7 GB), config và tokenizer. Hugging Face tự resume nếu cell bị ngắt.
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', '-q', 'install', 'huggingface_hub'], check=True)
from huggingface_hub import hf_hub_download

MODELS = REPO / 'test/deepseek-inference/models'
MODELS.mkdir(parents=True, exist_ok=True)
MODEL_REPO = 'bobbydeveloper2014/deepseek-coder-6.7b-nimq'
MODEL_REV = '2bd1dc094b4c8963232632aa79cdc4ef884efda1'
for name in ['config.json', 'tokenizer.json', 'tokenizer_config.json', 'model.nimq']:
    hf_hub_download(MODEL_REPO, filename=name, revision=MODEL_REV, local_dir=str(MODELS))
for f in sorted(MODELS.iterdir()):
    print(f'{f.name:24} {f.stat().st_size / 2**20:10.1f} MB')

config.json:   0%|          | 0.00/1.13k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.37M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.71k [00:00<?, ?B/s]

model.nimq: reconstructing file:   0%|          |  0.00B / 4.71GB            

model.nimq: downloading bytes:           |  0.00B            

.cache                          0.0 MB
config.json                     0.0 MB
config.nim.json                 0.0 MB
model.nimq                   4488.2 MB
tokenizer.json                  1.3 MB
tokenizer_config.json           0.0 MB


In [6]:
# 6) Biên dịch shader Vulkan và binary nim_inference.
import os
os.chdir(str(REPO))

!make shaders
!make build-nim-inference

[shaders] done.
nim check -d:release --hints:off --mm:arc --opt:speed --path:. --path:test/codestral --path:test/deepseek-inference --path:test/spectron --path:kernelfortest test/deepseek-inference/nim_inference.nim
/content/nimformers-framework/vendor/bybylang/probe_data.nim(2, 23) Warning: imported and not used: 'os' [UnusedImport]
/content/nimformers-framework/vendor/bybylang/backends/metal/metal_backend.nim(2, 12) Warning: imported and not used: 'os' [UnusedImport]
/content/nimformers-framework/vendor/bybylang/backends/metal/metal_backend.nim(3, 8) Warning: imported and not used: 'probe' [UnusedImport]
/content/nimformers-framework/vendor/bybylang/backends/vulkan/vulkan_driver.nim(1221, 5) Warning: use exitprocs.addExitProc; addQuitProc is deprecated [Deprecated]
/content/nimformers-framework/vendor/bybylang/gpubackend.nim(828, 5) Warning: unreachable else, all cases are already covered [UnreachableElse]
/content/nimformers-framework/test/deepseek-inference/nim_inference.nim(5, 27)

## Inference

`model.nimq` được mmap thẳng từ đĩa, không đọc cả file vào RAM. Trên T4 (dGPU) mỗi weight được copy một lần từ mmap vào VRAM. `BYBY_VULKAN_DEVICE_LOCAL=auto` đặt weights ở VRAM khi Vulkan nhận diện dGPU NVIDIA.

Lúc load, mỗi shape Q4 được autotune hai lần: M=1 cho decode và M=64 cho prefill. Dòng `[q4-tune:vulkan] ... M=64: KHR-cooperative-matrix` nghĩa là GEMM prefill của shape đó chạy trên Tensor Core. Dòng `[vulkan] ... matrix=KHR_cooperative_matrix FP16/FP32` xác nhận driver hỗ trợ. Trên dGPU dòng này có thêm `prefill=tensor-cores-only`: mọi GEMM prefill chỉ chạy kernel Tensor Core (`KHR-cooperative-matrix-tiled64` hoặc `KHR-cooperative-matrix`), không bao giờ chọn kernel scalar. Cuối cell có tóm tắt.

`lm_head` chỉ tính logits cho token cuối của prompt. Lần chạy đầu phải biên dịch shader bằng glslc nên chậm hơn; các lần sau dùng cache.

In [7]:
# 7) Chạy inference.
import os
from pathlib import Path

PROMPT = 'You are a senior software engineer. Write a complete Python program to manage a library of books that meets these requirements: (1) Define a Book class with attributes: id, title, author, year, available (bool). (2) Define a Library class holding a list of books, with methods: add_book, remove_book(id), find_by_author(author), borrow_book(id), return_book(id), and list_available(). (3) Handle exceptions when borrowing an already-borrowed or nonexistent book, printing clear error messages. (4) Write a main() function demonstrating adding 5 books, borrowing/returning, searching by author, and listing available books. (5) Ensure full type hints, docstrings for every class and method, and PEP8 compliance. (6) Add unit tests using unittest for at least 4 methods, including error cases. (7) Briefly explain the time complexity of each method and suggest improvements if the list holds millions of books. Present the code in separate markdown blocks for the main code and the tests, with explanations in English.'
MAX_NEW_TOKENS = 999
TEMPERATURE = 0.0          # 0 = greedy
TOP_P = 0.9
model = MODELS / 'model.nimq'
assert model.exists() and (MODELS / 'tokenizer.json').exists()

os.chdir(str(REPO))

print('Backend Vulkan | mmap | max tokens =', MAX_NEW_TOKENS)
print('model =', model)

!BYBY_MODEL_MMAP=1 \
 BYBY_VULKAN_DEVICE_LOCAL=auto \
 BYBY_VULKAN_TENSOR_CORES=1 \
 ./nim_inference --no-progress "{MODELS}" "{model}" vulkan "{PROMPT}" {MAX_NEW_TOKENS} {TEMPERATURE} {TOP_P}

Backend Vulkan | mmap | max tokens = 999
model = /content/nimformers-framework/test/deepseek-inference/models/model.nimq
Loading config...
Loading native tokenizer...
Init backend (vulkan)...
== [Framework] Backend selected: vulkan ==
Building model...
Loading weights tu /content/nimformers-framework/test/deepseek-inference/models/model.nimq ...
  reading file /content/nimformers-framework/test/deepseek-inference/models/model.nimq ...
  file size: 4488 MB
[load] read 4488 MB sequentially in 13.12 s (BYBY_PREFAULT=0 skips)
  mapped 515 tensors from the file (NIMQ2: the GPU reads payloads at arbitrary byte offsets in the mmap), building the model...
  loading layer 1/32 ...
  loading layer 2/32 ...
  loading layer 3/32 ...
  loading layer 4/32 ...
  loading layer 5/32 ...
  loading layer 6/32 ...
  loading layer 7/32 ...
  loading layer 8/32 ...
  loading layer 9/32 ...
  loading layer 10/32 ...
  loading layer 11/32 ...
  loading layer 12/32 ...
  loading layer 13/32 ...
  loading layer

## formers.n

Chạy cùng model bằng `formers-cli` (CLI kiểu llama.cpp của `other/formers.n`): nó tự nhận diện kiến trúc Llama từ `config.json` và header `.nimq`, rồi dùng các block của `nim_inference`. `--verbose` để thấy dòng `[vulkan] ... prefill=tensor-cores-only` và các dòng `[q4-tune:vulkan] ... M=64`; bỏ `--verbose` thì chỉ còn text sinh ra và `Speed stats`.

In [9]:
import os
os.chdir(str(REPO))

!make formers-cli

# Baseline: fused QKV
!BYBY_UBATCH=512 BYBY_PREFILL_CB_MS=300 BYBY_VULKAN_Q4_SPLIT_QKV=0 other/formers.n/formers-cli --models "{MODELS}" --backend vulkan -p "{PROMPT}" -n 255 --temp 0 --top-p {TOP_P} --no-progress --verbose

# A/B: three autotuned Q4 GEMVs
!BYBY_UBATCH=512 BYBY_PREFILL_CB_MS=300 BYBY_VULKAN_Q4_SPLIT_QKV=1 other/formers.n/formers-cli --models "{MODELS}" --backend vulkan -p "{PROMPT}" -n 255 --temp 0 --top-p {TOP_P} --no-progress --verbose

make -C other/formers.n build
make[1]: Entering directory '/content/nimformers-framework/other/formers.n'
make -C /content/nimformers-framework/vendor/bybylang build
make[2]: Entering directory '/content/nimformers-framework/vendor/bybylang'
[Makefile] OS: Linux
nim c -d:release -o:bybylang bybylang.nim
Hint: used config file '/content/tools/nim-2.2.10/config/nim.cfg' [Conf]
Hint: used config file '/content/tools/nim-2.2.10/config/config.nims' [Conf]
.............................................................................................................................................
/content/nimformers-framework/vendor/bybylang/probe_data.nim(2, 23) Warning: imported and not used: 'os' [UnusedImport]
......
/content/nimformers-framework/vendor/bybylang/backends/cpu/cpu_backend.nim(56, 6) Hint: 'nimf_cpu_current_isa' is declared but not used [XDeclaredButNotUsed]
.
/content/nimformers-framework/vendor/bybylang/backends/metal/metal_backend.nim(4, 6) Hint: 'c_free' is declared but 

### Trace và chạy lifted FIR trên Tesla T4

Cell đầu chạy runtime Vulkan dưới `firlift --run`: hai khối prefill và một bước decode được ghi qua pipe rồi lift thẳng thành `model.fir`. Không có file trace và không có dữ liệu upload nào được ghi ra đĩa. Weight Q4 được khai báo `q4r`: mỗi ma trận nằm trong một buffer riêng và được nạp nguyên byte từ `model.nimq`, còn activation và KV cache dùng chung một arena với context 512.

Cell kế tiếp chạy graph FIR vừa tạo trên cùng prompt, cùng model Q4, greedy, backend Vulkan. So sánh text với lần chạy native ở các cell trên; tốc độ của FIR graph được in riêng cho prefill và decode.

In [ ]:
# Lift: run the Vulkan runtime under firlift, record two prefill blocks and one decode step
# through a pipe and write model.fir. Nothing but the lifted graph is written to the disk.
import os
import re
import subprocess
import sys
import tempfile
from pathlib import Path

os.chdir(str(REPO))

FIRLIFT = REPO / "other/formers.n/firlift/firlift"
FIRLIFT_SOURCE = REPO / "other/formers.n/firlift/firlift_main.nim"
FORMERS_CLI = REPO / "other/formers.n/formers-cli"
FIR_OUT = Path("/content/deepseek-t4-lifted")
FIR_OUT.mkdir(parents=True, exist_ok=True)
FIR_MODEL = FIR_OUT / "model.fir"
FIR_CONTEXT = 512

MODEL = Path(model)
MODELS = Path(MODELS)

assert FORMERS_CLI.is_file(), (
    f"formers-cli not found: {FORMERS_CLI}. "
    "Run the formers-cli build cell first."
)
assert MODEL.is_file(), f"Model not found: {MODEL}"
assert (MODELS / "tokenizer.json").is_file(), "tokenizer.json not found"


def run_streaming(command, env, what):
    """Run a command, print its output line by line and return the whole output."""
    process = subprocess.Popen(
        [str(part) for part in command],
        cwd=str(REPO),
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        errors="replace",
    )
    lines = []
    for line in process.stdout:
        print(line, end="", flush=True)
        lines.append(line)
    code = process.wait()
    if code:
        raise RuntimeError(f"{what} failed: exit code {code}")
    return "".join(lines)


try:
    from tokenizers import Tokenizer
except ImportError:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "tokenizers"],
        check=True,
    )
    from tokenizers import Tokenizer

tokenizer = Tokenizer.from_file(str(MODELS / "tokenizer.json"))
prompt_ids = tokenizer.encode(PROMPT, add_special_tokens=False).ids

# The recorded prompt fills one 64-row prefill block and a 16-row one.
# The CLI adds one special token beyond the tokenizer count below.
TRACE_TOKENIZER_TARGET = 79
TRACE_RUNTIME_TARGET = 80
TRACE_PROMPT = None

for token_count in range(
    max(1, TRACE_TOKENIZER_TARGET - 8),
    min(len(prompt_ids), TRACE_TOKENIZER_TARGET + 8) + 1,
):
    candidate = tokenizer.decode(
        prompt_ids[:token_count],
        skip_special_tokens=False,
    )
    encoded_count = len(
        tokenizer.encode(candidate, add_special_tokens=True).ids
    )
    if encoded_count == TRACE_TOKENIZER_TARGET:
        TRACE_PROMPT = candidate
        break

if TRACE_PROMPT is None:
    raise ValueError(
        f"Could not construct a {TRACE_TOKENIZER_TARGET}-token trace prompt "
        f"from PROMPT ({len(prompt_ids)} ordinary tokens)."
    )

print(
    "Tokenizer trace prompt:",
    len(tokenizer.encode(TRACE_PROMPT, add_special_tokens=True).ids),
    "tokens; expected runtime count:",
    TRACE_RUNTIME_TARGET,
)

# Build firlift. formers-cli was built in the previous notebook cell.
nimcache = Path(tempfile.mkdtemp(prefix="nimcache-firlift-t4-"))
run_streaming(
    [
        "nim", "c", "-d:release", "--hints:off",
        "--path:vendor/bybylang",
        f"--nimcache:{nimcache}",
        f"-o:{FIRLIFT}",
        FIRLIFT_SOURCE,
    ],
    os.environ.copy(),
    "Building firlift",
)

# Record and lift in one process. Without --weights and without upload data the lifter names
# every weight after its checkpoint tensor (the Q4 matrices become resident q4r weights) and
# keeps activations and KV caches in one arena. The kernel choice is the lifter's default for
# a Vulkan trace, so no FIRLIFT_* override is passed.
lift_env = {
    key: value for key, value in os.environ.items()
    if not key.startswith("FIRLIFT_")
}
lift_env.update({
    "BYBY_UBATCH": "512",
    "BYBY_PREFILL_CB_MS": "450",
    "BYBY_FIR_PROBE_STEPS": "prefill:2,decode:1",
    "BYBY_FIR_PROBE_DATA": "0",
})
lift_env.pop("BYBY_FIR_PROBE", None)

lift_output = run_streaming(
    [
        FIRLIFT,
        "--config", MODELS / "config.json",
        "--context", FIR_CONTEXT,
        "--out", FIR_OUT,
        "--report",
        "--run",
        FORMERS_CLI,
        "--models", MODELS,
        "-m", MODEL,
        "--raw",
        "--add-special-tokens",
        "-p", TRACE_PROMPT,
        "-n", "4",
        "--temp", "0",
        "--top-p", TOP_P,
        "--backend", "vulkan",
        "--no-progress",
        "--no-warmup",
        "-c", FIR_CONTEXT,
    ],
    lift_env,
    "Recording and lifting the Vulkan run",
)

prefill_match = re.search(r"Prefill:\s*(\d+)\s*tokens", lift_output)
if not prefill_match:
    raise RuntimeError("Could not find the runtime prefill token count.")
if int(prefill_match.group(1)) != TRACE_RUNTIME_TARGET:
    raise RuntimeError(
        f"Runtime recorded {prefill_match.group(1)} prompt tokens; "
        f"expected {TRACE_RUNTIME_TARGET}."
    )

assert FIR_MODEL.is_file(), f"Lifted graph missing: {FIR_MODEL}"

# Show the phases and the weight declarations the runner will see.
fir_lines = FIR_MODEL.read_text().splitlines()
phase_lines = [line.strip() for line in fir_lines if line.startswith("phase ")]
weight_dtypes = {}
for line in fir_lines:
    words = line.split()
    if len(words) >= 3 and words[0] == "weight":
        weight_dtypes[words[2]] = weight_dtypes.get(words[2], 0) + 1
print("Lifted phases:")
for line in phase_lines:
    print(" ", line)
print("Lifted weights by dtype:", weight_dtypes)

if not any(dtype.startswith("q4r") for dtype in weight_dtypes):
    raise RuntimeError("The lifted graph declares no resident Q4 weights.")

single_row_phases = [
    line for line in phase_lines
    if re.search(r"\brows=1(?:\s|$)", line)
]
if single_row_phases:
    raise RuntimeError(
        "The lifted graph contains a one-row prefill phase; "
        "record a prompt whose last block has at least two rows."
    )

# Validate the graph before the weights are loaded.
run_streaming([FORMERS_CLI, "--fir-check", FIR_MODEL], os.environ.copy(), "FIR check")

In [ ]:
# Run the lifted graph on the full prompt (greedy, Vulkan).
import os

os.chdir(str(REPO))

FIR_TOKENS = 64
fir_env = os.environ.copy()
fir_env.update({
    "BYBY_UBATCH": "512",
    "BYBY_PREFILL_CB_MS": "450",
})

print(f"Running the lifted FIR graph on Vulkan for {FIR_TOKENS} tokens...")
run_streaming(
    [
        FORMERS_CLI,
        "--models", MODELS,
        "-m", MODEL,
        "--fir", FIR_MODEL,
        "--raw",
        "--add-special-tokens",
        "-p", PROMPT,
        "-n", FIR_TOKENS,
        "--temp", "0",
        "--top-p", TOP_P,
        "--backend", "vulkan",
        "--no-progress",
        "-c", FIR_CONTEXT,
        "--verbose",
    ],
    fir_env,
    "Lifted FIR run",
)

### Tài liệu

- [Nim 2.2.10](https://nim-lang.org/download/)
- [Google Colab FAQ: GPU runtime](https://research.google.com/colaboratory/faq.html)
- [VK_KHR_cooperative_matrix](https://registry.khronos.org/vulkan/specs/latest/man/html/VK_KHR_cooperative_matrix.html)
- [DeepSeek Coder 6.7B NimQ trên Hugging Face](https://huggingface.co/bobbydeveloper2014/deepseek-coder-6.7b-nimq)